# Analisi di dati testuali

I dati testuali sono dati non strutturati.

Esempi:

```text
recensioni
commenti
documenti
email
post
```

Un modello di machine learning non può lavorare direttamente su stringhe di testo.

Prima è necessario trasformare il testo in una rappresentazione numerica.

L'obiettivo è ottenere, per ogni documento, un vettore di feature numeriche.

Una delle rappresentazioni più semplici è il modello Bag-of-Words.

## Bag-of-Words

Il modello Bag-of-Words rappresenta un documento contando le parole che compaiono al suo interno.

Ogni parola del vocabolario diventa una feature.

Ogni documento diventa un vettore.

Il valore di una feature indica quante volte quella parola compare nel documento.

Esempio:

```text
documento 1: "good movie good"
documento 2: "bad movie"
```

Vocabolario:

```text
bad, good, movie
```

Rappresentazione:

```text
documento 1: [0, 2, 1]
documento 2: [1, 0, 1]
```

Il modello ignora l'ordine delle parole.

Quindi conserva l'informazione sulla frequenza dei termini, ma perde la struttura della frase.

## Due recensioni IMDb

Le due stringhe seguenti rappresentano due recensioni testuali.

Sono ancora dati non numerici.

Prima di usarle in un modello bisogna:

```text
pulire il testo
normalizzare le parole
trasformare i documenti in vettori numerici
```

In [7]:
text0 = "OK... so... I really like Kris Kristofferson and his usual easy going delivery of lines in his movies. Age has helped him with his soft spoken low energy style and he will steal a scene effortlessly. But, Disappearance is his misstep. Holy Moly, this was a bad movie! <br /><br />I must give kudos to the cinematography and and the actors, including Kris, for trying their darndest to make sense from this goofy, confusing story! None of it made sense and Kris probably didn't understand it either and he was just going through the motions hoping someone would come up to him and tell him what it was all about! <br /><br />I don't care that everyone on this movie was doing out of love for the project, or some such nonsense... I've seen low budget movies that had a plot for goodness sake! This had none, zilcho, nada, zippo, empty of reason... a complete waste of good talent, scenery and celluloid! <br /><br />I rented this piece of garbage for a buck, and I want my money back! I want my 2 hours back I invested on this Grade F waste of my time! Don't watch this movie, or waste 1 minute of your valuable time while passing through a room where it's playing or even open up the case that is holding the DVD! Believe me, you'll thank me for the advice!"

text1 = "This movie is a real gem. The performances are excellent, the story is moving, and the direction is careful and intelligent. I was really impressed by the atmosphere and by the way the characters are developed. It is not just entertainment, it is a touching and memorable film."

## Stemming

Lo stemming riduce le parole alla loro radice.

Esempio:

```text
playing  -> play
players  -> player
studies  -> studi
```

Lo scopo è ridurre varianti simili della stessa parola a una forma comune.

Questo diminuisce la dimensione del vocabolario.

Lo stemming è più semplice e grezzo della lemmatizzazione.

La lemmatizzazione cerca invece la forma base corretta della parola, ma è più costosa e richiede più informazione linguistica.

In [8]:
import nltk

## NLTK

`nltk` è una libreria per il Natural Language Processing.

Può essere usata per:

```text
tokenizzazione
stemming
lemmatizzazione
gestione di risorse linguistiche
```

Nel codice viene usato lo stemmer di Porter, uno degli algoritmi classici per lo stemming in inglese.

In [9]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/pc/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Pulizia del testo

La funzione `clean_and_stem` applica quattro passaggi:

```text
1. converte il testo in minuscolo
2. rimuove caratteri non alfabetici
3. divide il testo in parole
4. applica lo stemming
```

Il risultato è una lista di token normalizzati.

La rimozione dei caratteri non alfabetici elimina punteggiatura, numeri, tag HTML e simboli.

Lo stemming riduce parole simili a forme più compatte.

In [10]:
import re
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()

def clean_and_stem(text):
    # 1. lowercase
    text = text.lower()
    
    # 2. rimozione caratteri non alfabetici (pulizia "robusta")
    text = re.sub(r"[^a-zàèéìòù\s]", "", text)
    
    # 3. tokenizzazione semplice
    tokens = text.split()
    
    # 4. stemming
    return [porter.stem(w) for w in tokens]


text0 = "The players are playing and <the studies are connected to learning"
text1 = "The researchers are analyzing data and developing new models"

text0 = " ".join(clean_and_stem(text0))
text1 = " ".join(clean_and_stem(text1))

print(text0)
print(text1)

the player are play and the studi are connect to learn
the research are analyz data and develop new model


## Risultato del preprocessing

Dopo la pulizia:

```text
The players are playing
```

diventa una sequenza di token normalizzati.

Parole come:

```text
players
playing
```

vengono ridotte tramite stemming.

Il testo finale viene ricostruito con:

```python
" ".join(...)
```

per poter essere passato al vettorizzatore di `sklearn`.

## Vettorizzazione

La vettorizzazione trasforma documenti testuali in matrici numeriche.

`CountVectorizer` costruisce automaticamente:

```text
vocabolario
matrice Bag-of-Words
```

Il vocabolario associa ogni parola a una colonna.

La matrice contiene una riga per documento e una colonna per parola.

Il valore nella cella indica quante volte quella parola compare in quel documento.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

count = CountVectorizer()

docs = np.array([text0, text1])

X = count.fit_transform(docs).toarray()
print("Indici")
print(count.vocabulary_)
print("Matrice BoW")
print(X)

Indici
{'the': 13, 'player': 10, 'are': 2, 'play': 9, 'and': 1, 'studi': 12, 'connect': 3, 'to': 14, 'learn': 6, 'research': 11, 'analyz': 0, 'data': 4, 'develop': 5, 'new': 8, 'model': 7}
Matrice BoW
[[0 1 2 1 0 0 1 0 0 1 1 0 1 2 1]
 [1 1 1 0 1 1 0 1 1 0 0 1 0 1 0]]


## Vocabolario e matrice BoW

`count.vocabulary_` è un dizionario.

Le chiavi sono le parole.

I valori sono gli indici delle colonne nella matrice.

Se:

```python
count.vocabulary_["the"] = 8
```

allora la colonna 8 della matrice contiene il numero di occorrenze della parola `the` in ogni documento.

La matrice `X` ha forma:

```text
numero_documenti x numero_parole_del_vocabolario
```

In [12]:
col = count.vocabulary_['the']
print(col)
print(X[:, col])

13
[2 1]


## Term Frequency

La Term Frequency misura la frequenza relativa di una parola in un documento.

Partendo dalla matrice BoW, ogni riga contiene i conteggi delle parole in un documento.

Per trasformare i conteggi in frequenze relative, si divide ogni valore per il numero totale di termini del documento.

La formula è:

$$
TF(t, d) =
\frac{\text{conteggio del termine } t \text{ nel documento } d}
{\text{numero totale di termini nel documento } d}
$$

Così documenti di lunghezza diversa diventano più confrontabili.

In [13]:
X.sum(axis=1, keepdims=True) # somma sulle colonne. Per ogni riga, numero di termini 

array([[11],
       [ 9]])

In [14]:
# -----------------------
# TERM FREQUENCY (TF)
# -----------------------
# normalizzazione per documento (frequenza relativa)
TF = X / X.sum(axis=1, keepdims=True) 

print("Matrice TF")
print(TF)

Matrice TF
[[0.         0.09090909 0.18181818 0.09090909 0.         0.
  0.09090909 0.         0.         0.09090909 0.09090909 0.
  0.09090909 0.18181818 0.09090909]
 [0.11111111 0.11111111 0.11111111 0.         0.11111111 0.11111111
  0.         0.11111111 0.11111111 0.         0.         0.11111111
  0.         0.11111111 0.        ]]


## Interpretazione della matrice TF

Ogni riga della matrice TF rappresenta un documento.

La somma dei valori di ogni riga è pari a 1.

Una parola ha TF alta in un documento se compare spesso rispetto alla lunghezza del documento.

La TF riduce l'effetto della lunghezza dei documenti, ma non distingue ancora tra parole comuni e parole informative.

## Inverse Document Frequency

La IDF misura quanto un termine è raro nel corpus.

Se una parola compare in molti documenti, è probabilmente meno informativa.

Se una parola compare in pochi documenti, può essere più discriminante.

Nel codice:

```python
df = np.count_nonzero(X > 0, axis=0)
```

`df` indica in quanti documenti compare ogni parola.

La formula usata è:

$$
IDF(t) = \log\left(\frac{N}{df(t)}\right)
$$

dove:

- $N$ è il numero totale di documenti;
- $df(t)$ è il numero di documenti che contengono il termine $t$.

In [15]:
np.count_nonzero(X, axis=0) # per ogni colonna, conta quante righe non sono nulle ovvero in quante righe compare il termine

array([1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1])

In [16]:
N = X.shape[0]
df = np.count_nonzero(X > 0, axis=0)
IDF = np.log(N / df)

print("Vettore IDF:")
print(IDF)

Vettore IDF:
[0.69314718 0.         0.         0.69314718 0.69314718 0.69314718
 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718 0.69314718
 0.69314718 0.         0.69314718]


## Interpretazione della IDF

Se un termine compare in tutti i documenti:

$$
df(t) = N
$$

allora:

$$
IDF(t) = \log\left(\frac{N}{N}\right) = 0
$$

Il termine viene considerato poco informativo.

Se un termine compare in pochi documenti, allora:

$$
df(t)
$$

è piccolo e il rapporto:

$$
\frac{N}{df(t)}
$$

è maggiore.

Quindi l'IDF aumenta.

In [17]:
TFIDF = TF * IDF

print("Matrice TF-IDF:")
print(TFIDF)

Matrice TF-IDF:
[[0.         0.         0.         0.06301338 0.         0.
  0.06301338 0.         0.         0.06301338 0.06301338 0.
  0.06301338 0.         0.06301338]
 [0.07701635 0.         0.         0.         0.07701635 0.07701635
  0.         0.07701635 0.07701635 0.         0.         0.07701635
  0.         0.         0.        ]]


## TF-IDF

La TF-IDF combina:

```text
TF:
quanto una parola è frequente in un documento

IDF:
quanto una parola è rara nel corpus
```

La formula è:

$$
TFIDF(t,d) = TF(t,d) \cdot IDF(t)
$$

Una parola ottiene peso alto se:

```text
compare spesso in un documento
ma non compare in molti documenti
```

Una parola ottiene peso basso se:

```text
compare spesso
ma compare anche in molti documenti
```

Quindi la TF-IDF riduce il peso delle parole comuni e aumenta il peso delle parole più informative.

## Dataset IMDb

Il dataset `movie_data.csv` contiene recensioni cinematografiche e sentiment associato.

La prima colonna contiene il testo della recensione.

La seconda colonna contiene l'etichetta di sentiment.

Nel codice vengono usati solo i primi 5000 esempi, perché il dataset completo può essere più pesante da elaborare.

In [21]:
import pandas as pd
import os

df = pd.read_csv(os.path.join('materiale_2026/dataset/', "movie_data.csv"))

X = df.iloc[:5000, 0]  # recensioni   solo una parte, ds troppo grande
y = df.iloc[:5000, 1]  # sentiment

In [22]:
X

0       In 1974, the teenager Martha Moxley (Maggie Gr...
1       OK... so... I really like Kris Kristofferson a...
2       ***SPOILER*** Do not read this, if you think a...
3       hi for all the people who have seen this wonde...
4       I recently bought the DVD, forgetting just how...
                              ...                        
4995    "L'Ossessa" (released in English under many ti...
4996    I'm a nice guy, and I like to think of myself ...
4997    Being a Russian myself, sometimes it's hard fo...
4998    Forbidden Planet rates as landmark in science ...
4999    I didn't understand the people who rated it ov...
Name: review, Length: 5000, dtype: object

## Preprocessing del dataset

Ogni recensione viene pulita e trasformata tramite `clean_and_stem`.

La lista di token viene poi ricostruita come stringa.

Questo serve perché `CountVectorizer` si aspetta una sequenza di documenti testuali.

Il risultato è un array di recensioni già preprocessate.

In [23]:
count = CountVectorizer()

X = np.array([" ".join(clean_and_stem(doc)) for doc in X])

X = count.fit_transform(X).toarray()

## Matrice Bag-of-Words del dataset

Dopo `fit_transform`, `X` non contiene più testi.

Contiene una matrice numerica Bag-of-Words.

Ogni riga rappresenta una recensione.

Ogni colonna rappresenta una parola del vocabolario.

Il valore indica quante volte quella parola compare nella recensione.

In [24]:
TF = X / X.sum(axis=1, keepdims=True)
N = X.shape[0]
df = np.count_nonzero(X > 0, axis=0)
IDF = np.log(N / df)
TFIDF = TF * IDF

## TF-IDF sul dataset IMDb

La matrice Bag-of-Words viene trasformata in TF-IDF.

Il risultato `TFIDF` è la rappresentazione finale usata dal modello.

Ogni recensione è rappresentata da un vettore numerico.

Le parole comuni in molte recensioni ricevono peso basso.

Le parole più specifiche ricevono peso più alto.

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    TFIDF, y, test_size=0.2, random_state=8
)

## Classificazione del sentiment

Il modello riceve come input la rappresentazione TF-IDF delle recensioni.

La Logistic Regression impara ad associare certi pattern di parole al sentiment.

Il training set viene usato per addestrare il modello.

Il test set viene usato per valutare l'accuracy su recensioni non viste.

In [26]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [27]:
y_pred = model.predict(X_test)

print("Accuracy: ", np.mean(y_pred == y_test))

Accuracy:  0.839


## Costo computazionale

Sia:

- $n$ il numero di documenti;
- $V$ la dimensione del vocabolario;
- $L$ la lunghezza media dei documenti.

Il preprocessing ha costo proporzionale al numero totale di token:

$$
O(nL)
$$

La costruzione della Bag-of-Words ha costo circa:

$$
O(nL)
$$

La matrice BoW o TF-IDF ha dimensione:

$$
n \times V
$$

Se viene convertita in array denso con:

```python
.toarray()
```

la memoria richiesta diventa:

$$
O(nV)
$$

Questo può diventare molto pesante quando il vocabolario è grande.

Per dataset testuali reali è spesso meglio mantenere la matrice in formato sparso.